# 01 · Data Ingestion Engine – Architecture Test

> **PyHydroGeophysiX** · Macro-Area 1 · Layer 1
>
> This notebook verifies that `src/data_ingestion.py` is correctly wired and
> that every connector can round-trip a synthetic raw file through the
> factory pattern.

---

## Contents

1. [Environment check](#1-environment-check)  
2. [Create synthetic raw files](#2-create-synthetic-raw-files)  
3. [Test ConnectorFactory](#3-test-connectorfactory)  
4. [Verify DataFrame contract](#4-verify-dataframe-contract)  
5. [Error-handling smoke-tests](#5-error-handling-smoke-tests)  
6. [Summary](#6-summary)  

## 1 · Environment check

In [ ]:
import sys
import importlib

print(f"Python : {sys.version}")

# ── Make sure 'src' is on the path (works when the notebook is run from
# ── 'scripts/' or from the project root).
from pathlib import Path

# Resolve project root robustly
_this_nb   = Path().resolve()                    # CWD when running
_proj_root = _this_nb.parent if _this_nb.name == 'scripts' else _this_nb

if str(_proj_root) not in sys.path:
    sys.path.insert(0, str(_proj_root))

print(f"Project root : {_proj_root}")
print(f"sys.path[0]  : {sys.path[0]}")

In [ ]:
# ── Import the engine ──────────────────────────────────────────────────────
from src.data_ingestion import (
    BaseConnector,
    MeterConnector,
    ERTConnector,
    MeteoConnector,
    CosmosConnector,
    ConnectorFactory,
)

print("✓ src.data_ingestion imported successfully.")
print(f"  Available connectors: {ConnectorFactory.available_connectors()}")

## 2 · Create synthetic raw files

We generate minimal CSV fixtures in `data/raw/` so that every connector has
something to chew on — no real sensor hardware required.

In [ ]:
import pandas as pd
import numpy as np

RAW_DIR = _proj_root / 'data' / 'raw'
RAW_DIR.mkdir(parents=True, exist_ok=True)

# ── Shared date range (hourly, 24 steps) ──────────────────────────────────
dates = pd.date_range('2024-06-01', periods=24, freq='h')

rng = np.random.default_rng(seed=42)

# ── 1. Meter CSV ──────────────────────────────────────────────────────────
meter_df = pd.DataFrame({
    'timestamp'     : dates.strftime('%Y-%m-%dT%H:%M:%S'),
    'water_level_m' : rng.uniform(0.5, 2.0, size=24).round(4),
    'temperature_c' : rng.uniform(10,  18,  size=24).round(2),
})
meter_path = RAW_DIR / 'dummy_meter.csv'
meter_df.to_csv(meter_path, index=False)
print(f"✓ Meter CSV  → {meter_path}")

# ── 2. ERT CSV ────────────────────────────────────────────────────────────
n_quad = 24
ert_df = pd.DataFrame({
    'datetime'               : dates.strftime('%Y-%m-%dT%H:%M:%S'),
    'electrode_a'            : rng.integers(1, 5, size=n_quad),
    'electrode_b'            : rng.integers(5, 9, size=n_quad),
    'electrode_m'            : rng.integers(9, 13, size=n_quad),
    'electrode_n'            : rng.integers(13, 17, size=n_quad),
    'resistance_ohm'         : rng.uniform(5,  50,  size=n_quad).round(4),
    'apparent_resistivity_ohmm': rng.uniform(10, 200, size=n_quad).round(2),
})
ert_path = RAW_DIR / 'dummy_ert.csv'
ert_df.to_csv(ert_path, index=False)
print(f"✓ ERT  CSV  → {ert_path}")

# ── 3. Meteo CSV ──────────────────────────────────────────────────────────
meteo_df = pd.DataFrame({
    'obs_time'          : dates.strftime('%Y-%m-%dT%H:%M:%S'),
    'air_temp_c'        : rng.uniform(5,  30,  size=24).round(2),
    'rel_humidity_pct'  : rng.uniform(40, 100, size=24).round(1),
    'precip_mm'         : rng.uniform(0,   5,  size=24).round(2),
    'wind_speed_ms'     : rng.uniform(0,  10,  size=24).round(2),
    'solar_rad_wm2'     : rng.uniform(0, 800,  size=24).round(1),
})
meteo_path = RAW_DIR / 'dummy_meteo.csv'
meteo_df.to_csv(meteo_path, index=False)
print(f"✓ Meteo CSV → {meteo_path}")

# ── 4. COSMOS CSV ─────────────────────────────────────────────────────────
cosmos_df = pd.DataFrame({
    'utc_time'               : dates.strftime('%Y-%m-%dT%H:%M:%S'),
    'raw_neutron_count'      : rng.integers(1200, 1800, size=24),
    'corrected_neutron_count': rng.integers(1150, 1750, size=24),
    'soil_moisture_vol'      : rng.uniform(0.1,  0.5,  size=24).round(4),
})
cosmos_path = RAW_DIR / 'dummy_cosmos.csv'
cosmos_df.to_csv(cosmos_path, index=False)
print(f"✓ COSMOS CSV→ {cosmos_path}")

## 3 · Test ConnectorFactory

In [ ]:
# ── Verify factory instantiation for every registered key ─────────────────
for key, expected_cls in [
    ('meter',  MeterConnector),
    ('ert',    ERTConnector),
    ('meteo',  MeteoConnector),
    ('cosmos', CosmosConnector),
]:
    connector = ConnectorFactory.get(key)
    assert isinstance(connector, expected_cls), (
        f"Expected {expected_cls.__name__}, got {type(connector).__name__}"
    )
    print(f"  ✓  '{key}' → {connector}")

print("\nAll factory assertions passed.")

In [ ]:
# ── Verify KeyError on unknown key ────────────────────────────────────────
try:
    ConnectorFactory.get('radar')
    print("FAIL – expected KeyError was NOT raised.")
except KeyError as exc:
    print(f"✓ KeyError raised correctly: {exc}")

## 4 · Verify DataFrame contract

Each connector must return:
- a `pd.DataFrame`
- with a `DatetimeIndex` named `'datetime_utc'`
- that is timezone-aware (`UTC`)
- with **no empty columns**

In [ ]:
import pytz

test_cases = [
    ('meter',  meter_path),
    ('ert',    ert_path),
    ('meteo',  meteo_path),
    ('cosmos', cosmos_path),
]

results: dict[str, pd.DataFrame] = {}

for sensor_key, path in test_cases:
    print(f"\n{'─'*60}")
    print(f"  Sensor : {sensor_key}")
    print(f"  File   : {path.name}")

    connector = ConnectorFactory.get(sensor_key)
    df = connector.parse_data(path)

    # ── Contract assertions ──────────────────────────────────────────────
    assert isinstance(df, pd.DataFrame),           "Must return a DataFrame"
    assert isinstance(df.index, pd.DatetimeIndex), "Index must be DatetimeIndex"
    assert df.index.name == 'datetime_utc',        "Index must be named 'datetime_utc'"
    assert df.index.tz is not None,                "Index must be timezone-aware"
    assert str(df.index.tz) == 'UTC',              f"Expected UTC, got {df.index.tz}"
    assert len(df.columns) > 0,                   "DataFrame must have at least one column"
    assert len(df) == 24,                         f"Expected 24 rows, got {len(df)}"

    print(f"  Rows   : {len(df)}")
    print(f"  TZ     : {df.index.tz}")
    print(f"  Cols   : {df.columns.tolist()}")
    print(f"  ✓ All contract assertions passed.")

    results[sensor_key] = df

print(f"\n{'='*60}")
print("All connectors passed the DataFrame contract check.")

In [ ]:
# ── Quick peek at each result ─────────────────────────────────────────────
for key, df in results.items():
    print(f"\n── {key.upper()} (first 3 rows) ──")
    display(df.head(3))

## 5 · Error-handling smoke-tests

In [ ]:
# ── FileNotFoundError ─────────────────────────────────────────────────────
connector = ConnectorFactory.get('meter')

try:
    connector.parse_data(Path('data/raw/does_not_exist.csv'))
    print("FAIL – expected FileNotFoundError was NOT raised.")
except FileNotFoundError as exc:
    print(f"✓ FileNotFoundError raised correctly:\n  {exc}")

In [ ]:
# ── ValueError on wrong column name ───────────────────────────────────────
import tempfile, os

# Write a CSV with a wrong datetime column name
bad_csv = RAW_DIR / '_bad_meter.csv'
pd.DataFrame({'wrong_col': ['2024-01-01'], 'value': [1.0]}).to_csv(bad_csv, index=False)

try:
    connector.parse_data(bad_csv)
    print("FAIL – expected ValueError was NOT raised.")
except ValueError as exc:
    print(f"✓ ValueError raised correctly:\n  {exc}")
finally:
    bad_csv.unlink(missing_ok=True)

## 6 · Summary

| Connector | Class | File parsed | Rows | Timezone |
|-----------|-------|-------------|------|----------|
| `meter`  | `MeterConnector`  | `dummy_meter.csv`  | 24 | UTC |
| `ert`    | `ERTConnector`    | `dummy_ert.csv`    | 24 | UTC |
| `meteo`  | `MeteoConnector`  | `dummy_meteo.csv`  | 24 | UTC |
| `cosmos` | `CosmosConnector` | `dummy_cosmos.csv` | 24 | UTC |

**All architecture requirements verified ✓**

- Abstract base class `BaseConnector` enforces the `parse_data` contract  
- `ConnectorFactory` dispatches correctly by string key  
- Every DataFrame has a UTC-aware `DatetimeIndex` named `datetime_utc`  
- `FileNotFoundError` and `ValueError` are raised and logged appropriately  

> **Next step**: Layer 2 – Time resampling & gap-filling (`src/data_preprocessing.py`)